In [ ]:
import time
import itertools
import numpy as np
from math import factorial

## Замер времени

In [2]:
def measure_time(func, *args):
    start = time.perf_counter()
    result = func(*args)
    elapsed = (time.perf_counter() - start) * 1000  # мс
    return result, elapsed

## Задание 1

### Алгоритм Нарайана

In [3]:
def narayana_permutations(n):
    
    if n == 0:
        return []
    perm = list(range(1, n + 1))
    result = [perm[:]]
    while True:
        i = n - 2
        while i >= 0 and perm[i] >= perm[i + 1]:
            i -= 1
        if i < 0:
            break
        j = n - 1
        while perm[j] <= perm[i]:
            j -= 1
        perm[i], perm[j] = perm[j], perm[i]
        perm[i + 1:] = perm[i + 1:][::-1]
        result.append(perm[:])
    return result

### Алгоритм Джонсона-Троттера

In [4]:
def johnson_trotter_permutations(n):
    
    if n == 0:
        return []
    elements = list(range(1, n + 1))
    directions = [-1] * n
    result = [elements[:]]

    def is_mobile(i):
        d = directions[i]
        ni = i + d
        return 0 <= ni < n and elements[i] > elements[ni]

    while True:
        mobile_val = -1
        mobile_idx = -1
        for i in range(n):
            if is_mobile(i) and elements[i] > mobile_val:
                mobile_val = elements[i]
                mobile_idx = i
        if mobile_idx == -1:
            break
        
        ni = mobile_idx + directions[mobile_idx]
        elements[mobile_idx], elements[ni] = elements[ni], elements[mobile_idx]
        directions[mobile_idx], directions[ni] = directions[ni], directions[mobile_idx]
        mobile_idx = ni
        
        for i in range(n):
            if elements[i] > mobile_val:
                directions[i] = -directions[i]
        result.append(elements[:])
    return result

### Алгоритм вектора инверсий

In [5]:
def inversion_vector_permutations(n):
    
    if n == 0:
        return []
    result = []
    total = factorial(n)
    for k in range(total):
        inv = []
        num = k
        for i in range(1, n + 1):
            inv.append(num % i)
            num //= i
        inv.reverse()
        available = list(range(1, n + 1))
        perm = []
        for idx in inv:
            perm.append(available.pop(idx))
        result.append(perm)
    return result

## Задание 2

In [6]:
def unique_permutations(sequence):

    elements = sorted(sequence)
    result = []

    def backtrack(current, remaining):
        if not remaining:
            result.append(current[:])
            return
        seen = set()
        for i in range(len(remaining)):
            if remaining[i] not in seen:
                seen.add(remaining[i])
                current.append(remaining[i])
                backtrack(current, remaining[:i] + remaining[i + 1:])
                current.pop()

    backtrack([], elements)
    return result

## Задание 3

In [7]:
def all_subsets(elements):

    result = []
    n = len(elements)
    for r in range(1, n + 1):
        for combo in itertools.combinations(elements, r):
            result.append(list(combo))
    return result

## Задание 4

In [8]:
def stationery_purchase(budget, shopping_list, price_list):

    available = [
        (name, qty, price_list[name])
        for name, qty in shopping_list
        if name in price_list
    ]
    available.sort(key=lambda x: x[2])

    selected = {}
    remaining_budget = budget

    for name, qty_needed, price in available:
        if price <= remaining_budget:
            selected[name] = 1
            remaining_budget -= price

    for name, qty_needed, price in available:
        if name in selected:
            while selected[name] < qty_needed and price <= remaining_budget:
                selected[name] += 1
                remaining_budget -= price

    spent = budget - remaining_budget
    return selected, spent

## Запуск заданий

In [9]:
print("=" * 75)
print("ЗАДАНИЕ 1: Сравнение алгоритмов генерации перестановок (n = 1..10)")
print("=" * 75)

test_ns = list(range(1, 11))
t_narayana = []
t_johnson = []
t_inversion = []

header = (f"{'n':^5} | {'n!':^10} | "
          f"{'Нарайана (мс)':^16} | {'Джонсон-Троттер (мс)':^22} | "
          f"{'Вектор инверсий (мс)':^22}")
print(header)
print("-" * len(header))

for n in test_ns:
    _, t1 = measure_time(narayana_permutations, n)
    _, t2 = measure_time(johnson_trotter_permutations, n)
    _, t3 = measure_time(inversion_vector_permutations, n)
    t_narayana.append(t1)
    t_johnson.append(t2)
    t_inversion.append(t3)
    print(f"{n:^5} | {factorial(n):^10} | {t1:^16.4f} | {t2:^22.4f} | {t3:^22.4f}")

print()
print(f"Средн. время (мс): Нарайана={np.mean(t_narayana):.4f}  "
      f"Джонсон-Троттер={np.mean(t_johnson):.4f}  "
      f"Вектор инверсий={np.mean(t_inversion):.4f}")

print()
print("=" * 75)
print("ЗАДАНИЕ 2: Уникальные перестановки (с повторяющимися элементами)")
print("=" * 75)

task2_cases = [
    [1, 2, 1],
    [1, 2, 3],
    [1, 1, 1],
    [1, 2, 2, 3],
]

for seq in task2_cases:
    perms, t = measure_time(unique_permutations, seq)
    print(f"\nПоследовательность: {''.join(map(str, seq))}  "
          f"-> {len(perms)} перестановок  (время: {t:.4f} мс)")
    for p in perms:
        print(f"   {''.join(map(str, p))}")

print()
print("Время для последовательностей разной длины:")
print(f"{'Длина':^8} | {'Кол-во перест.':^16} | {'Время (мс)':^14}")
print("-" * 45)
for length in range(1, 9):
    seq = list(range(1, length + 1))
    perms, t = measure_time(unique_permutations, seq)
    print(f"{length:^8} | {len(perms):^16} | {t:^14.4f}")

print()
print("=" * 75)
print("ЗАДАНИЕ 3: Все возможные выборки из n элементов")
print("=" * 75)

items = ["стол", "стул", "шкаф"]
subsets, t = measure_time(all_subsets, items)
print(f"\nЭлементы: {items}")
print(f"Всего вариантов выборки: {len(subsets)}  (время: {t:.4f} мс)")
for s in subsets:
    print(f"   {s}")

print()
print("Время для разного числа элементов:")
print(f"{'n':^6} | {'Кол-во выборок':^16} | {'Время (мс)':^14}")
print("-" * 42)
t_subsets = []
subset_ns = list(range(1, 16))
for n in subset_ns:
    elems = list(range(1, n + 1))
    subs, t = measure_time(all_subsets, elems)
    t_subsets.append(t)
    print(f"{n:^6} | {len(subs):^16} | {t:^14.4f}")


print()
print("=" * 75)
print("ЗАДАНИЕ 4: Покупка канцелярских принадлежностей студента Петрова П.П.")
print("=" * 75)

budget = 500.0
shopping_list = [
    ("Ручка",     3),
    ("Тетрадь",   2),
    ("Карандаш",  5),
    ("Линейка",   1),
    ("Ластик",    2),
    ("Скрепки",   1),
    ("Папка",     1),
]
price_list = {
    "Ручка":     30.0,
    "Тетрадь":   80.0,
    "Карандаш":  15.0,
    "Линейка":   50.0,
    "Ластик":    20.0,
    "Скрепки":   25.0,
    "Папка":    120.0,
    "Степлер":  200.0,
}

print(f"\nБюджет студента: {budget:.2f} руб.")
print("\nСписок необходимых покупок:")
print(f"  {'Наименование':<15} | {'Нужно, шт.':^12} | {'Цена/шт., руб.':^16}")
print("  " + "-" * 48)
for name, qty in shopping_list:
    price = price_list.get(name, "—")
    price_str = f"{price:.2f}" if isinstance(price, float) else price
    print(f"  {name:<15} | {qty:^12} | {price_str:^16}")

(selected, spent), t_task4 = measure_time(stationery_purchase, budget, shopping_list, price_list)

print(f"\nРезультат (время поиска: {t_task4:.4f} мс):")
print(f"  {'Наименование':<15} | {'Куплено, шт.':^14} | {'Сумма, руб.':^14}")
print("  " + "-" * 48)
for name, qty in selected.items():
    total_price = qty * price_list[name]
    print(f"  {name:<15} | {qty:^14} | {total_price:^14.2f}")
print("  " + "-" * 48)
print(f"  Куплено наименований: {len(selected)} из {len(shopping_list)}")
print(f"  Потрачено: {spent:.2f} руб.  |  Остаток: {budget - spent:.2f} руб.")


ЗАДАНИЕ 1: Сравнение алгоритмов генерации перестановок (n = 1..10)
  n   |     n!     |  Нарайана (мс)   |  Джонсон-Троттер (мс)  |  Вектор инверсий (мс) 
---------------------------------------------------------------------------------------
  1   |     1      |      0.0041      |         0.0051         |         0.0076        
  2   |     2      |      0.0047      |         0.0088         |         0.0089        
  3   |     6      |      0.0113      |         0.0181         |         0.0102        
  4   |     24     |      0.0201      |         0.0381         |         0.0281        
  5   |    120     |      0.0750      |         0.1991         |         0.1580        
  6   |    720     |      1.2153      |         1.3352         |         1.1132        
  7   |    5040    |      3.3384      |        10.6703         |         8.6820        
  8   |   40320    |     78.1666      |        100.4209        |        105.8404       
  9   |   362880   |     623.2790     |       1487.12